In [104]:
import numpy as np
import random
import math

### Utils

In [105]:
# --- Safe Operations ---

EPS = 1e-6  # small constant to avoid divide-by-zero and overflow

def safe_div(a, b):
    """Safe division: returns a / b if |b| > EPS, otherwise returns 0."""
    try:
        return a / (b if abs(b) > EPS else EPS)
    except Exception:
        return 0.

def safe_log(a):
    """Safe natural log: returns log(|a|) if a is not too close to zero, otherwise returns 0."""
    if abs(a) < EPS:
        return 0.
    return np.log(abs(a))

def safe_exp(a):
    # Clip values to avoid overflow, e.g., limit a's absolute value
    a = np.clip(a, -150, 150)
    return np.exp(a)


# --- Primitive Set ---

# Define a set of functions that can operate elementwise on numpy arrays
primitive_set = {
    'add': {'func': np.add, 'arity': 2, 'repr': '+'},
    'sub': {'func': np.subtract, 'arity': 2, 'repr': '-'},
    'mul': {'func': np.multiply, 'arity': 2, 'repr': '*'},
    'div': {'func': safe_div, 'arity': 2, 'repr': '/'},
    'sin': {'func': np.sin, 'arity': 1, 'repr': 'sin'},
    'cos': {'func': np.cos, 'arity': 1, 'repr': 'cos'},
    'tan': {'func': np.tan, 'arity': 1, 'repr': 'tan'},
    'exp': {'func': safe_exp, 'arity': 1, 'repr': 'exp'},
    'log': {'func': safe_log, 'arity': 1, 'repr': 'log'},
    # Add other functions as needed...
}

# We also treat the input variables as terminals. In our case, since the shape of x is (n_vars, n_samples), we refer to them by index, e.g. x[0].
terminal_set = ['x[%d]' % i for i in range(10)]  # allocate up to 10 possible variables, will use as many as needed
# Optionally, add constants as terminals:
constant_pool = [str(round(random.uniform(-5, 5), 2)) for _ in range(5)]

### Node Structure

In [106]:
# --- Expression Tree Structure ---

class Node:
    def __init__(self, content, children=None):
        self.content = content  # either a key for primitive_set or a terminal (variable or constant)
        self.children = children if children is not None else []

    def is_terminal(self):
        return len(self.children) == 0

    def evaluate(self, x):
        """Recursively evaluate the expression tree.
           x is expected to be a numpy array where each row is an input variable."""
        # If node is a terminal, interpret it.
        if self.is_terminal():
            # If the content is of form 'x[i]', extract the row.
            if isinstance(self.content, str) and self.content.startswith('x['):
                # Evaluate expression, e.g., "x[0]" returns the first row
                index = int(self.content[2:-1])
                return x[index]
            else:
                # It is a constant string, convert to float and return a numpy array of constant
                return float(self.content) * np.ones(x.shape[1])
        else:
            # Get the function from the primitive set.
            op = primitive_set[self.content]
            # Recursively evaluate children:
            args = [child.evaluate(x) for child in self.children]
            # Apply the operator in a vectorized way.
            return op['func'](*args)

    def __str__(self):
        """Return a string representation of the expression."""
        if self.is_terminal():
            return str(self.content)
        else:
            op_repr = primitive_set[self.content]['repr']
            if primitive_set[self.content]['arity'] == 1:
                return f"{op_repr}({self.children[0]})"
            elif primitive_set[self.content]['arity'] == 2:
                return f"({self.children[0]} {op_repr} {self.children[1]})"
            else:
                return f"{self.content}(" + ", ".join(str(child) for child in self.children) + ")"

### Evolutionary Algorithm

In [107]:
# --- Generate Random Expression Trees ---

def generate_random_tree(max_depth, current_depth=0, n_vars=2):
    """Generate a random expression tree with a maximum depth."""
    if current_depth >= max_depth or (current_depth > 0 and random.random() < 0.2):
        # Terminal: either a variable (if within dimension) or a constant
        if random.random() < 0.7:
            # Choose one of the available input variables
            var_index = random.randint(0, n_vars - 1)
            return Node(f'x[{var_index}]')
        else:
            # Use a random constant from the pool
            return Node(random.choice(constant_pool))
    else:
        # Choose a random primitive operation from the set
        op_key = random.choice(list(primitive_set.keys()))
        arity = primitive_set[op_key]['arity']
        children = [generate_random_tree(max_depth, current_depth + 1, n_vars) for _ in range(arity)]
        return Node(op_key, children)

In [108]:
# --- Genetic Operators: Crossover and Mutation ---

def mutate(node, max_depth, n_vars):
    """Mutate a tree node by randomly replacing a subtree."""
    if random.random() < 0.1:  # mutation probability at this node level
        return generate_random_tree(max_depth, n_vars=n_vars)
    if not node.is_terminal():
        new_children = [mutate(child, max_depth, n_vars) for child in node.children]
        return Node(node.content, new_children)
    return node

def crossover(node1, node2):
    """Swap a random subtree between node1 and node2."""
    if random.random() < 0.1:
        return node2
    if not node1.is_terminal() and not node2.is_terminal():
        new_children = []
        for child in node1.children:
            # With a certain probability, replace this child with a randomly chosen subtree from node2.
            if random.random() < 0.5:
                new_children.append(random.choice(flatten_tree(node2)))
            else:
                new_children.append(child)
        return Node(node1.content, new_children)
    return node1

def flatten_tree(node):
    """Return a list of all nodes in the tree."""
    nodes = [node]
    if not node.is_terminal():
        for child in node.children:
            nodes.extend(flatten_tree(child))
    return nodes

In [109]:
# --- Fitness Function ---

def fitness(individual: Node, x: np.ndarray, y: np.ndarray):
    """Compute the Mean Squared Error (MSE) of an individual's expression on the data."""
    try:
        y_pred = individual.evaluate(x)
        # Ensure y_pred has the same shape as y.
        if y_pred.shape != y.shape:
            return np.inf
        mse = np.mean((y - y_pred)**2)
        if np.isnan(mse):
            return np.inf
        return mse
    except Exception:
        return np.inf

In [110]:
# --- Evolutionary Algorithm ---

def genetic_programming(x: np.ndarray, y: np.ndarray, n_vars: int, 
                        population_size=100, generations=500, max_depth=5, threshold=0.00001):
    # Create an initial population of random trees.
    population = [generate_random_tree(max_depth, n_vars=n_vars) for _ in range(population_size)]
    best_individual = None
    best_fitness = np.inf

    for gen in range(generations):
        # Evaluate fitness for each individual.
        fitness_values = [fitness(ind, x, y) for ind in population]

        # Check if we have a solution that is good enough.
        if min(fitness_values) < threshold:
            best_fitness = min(fitness_values)
            best_individual = population[fitness_values.index(best_fitness)]
            print(f"Early stopping at generation {gen}: best fitness = {best_fitness}")
            break
        
        # Save the best so far.
        for ind, fit in zip(population, fitness_values):
            if fit < best_fitness:
                best_fitness = fit
                best_individual = ind
        
        if gen % 100 == 0 or gen == generations - 1:
            print(f"Generation {gen}: best fitness = {best_fitness}")

        # Create a new population
        new_population = []
        for _ in range(population_size):
            # Tournament selection: choose two individuals and pick the better one.
            candidates = random.sample(list(zip(population, fitness_values)), 3)
            parent = min(candidates, key=lambda item: item[1])[0]
            # Clone and mutate
            child = mutate(parent, max_depth, n_vars)
            # Optionally, perform crossover with another individual.
            mate = random.choice(population)
            child = crossover(child, mate)
            new_population.append(child)
        population = new_population

    print("Best expression found:")
    print(best_individual)
    return best_individual

### Main Routine

In [111]:
# --- Main Routine for Each Dataset ---

def run_symbolic_regression(dataset_path: str, population_size=100, generations=50, max_depth=5):
    data = np.load(dataset_path)
    x = data['x']  # x shape: (n_vars, n_samples)
    y = data['y']  # y shape: (n_samples,)
    n_vars = x.shape[0]
    best_tree = genetic_programming(x, y, n_vars, population_size, generations, max_depth)
    # Convert the best tree to a Python expression string.
    expression_str = str(best_tree)
    return expression_str

In [112]:
"""
# --- Generate the 8 Functions ---

functions = {}
for i in range(3):
    dataset_file = f"data/problem_{i}.npz"
    print(f"Processing dataset {i} from {dataset_file}...")
    expr = run_symbolic_regression(dataset_file, population_size=100, generations=500, max_depth=5)
    # You might want to post-process the expression for aesthetics.
    func_def = f"def f{i+1}(x: np.ndarray) -> np.ndarray:\n"
    func_def += f"    return {expr}\n"
    functions[f'f{i+1}'] = func_def

# Print the functions
print("\nGenerated Functions:")
for key, func_str in functions.items():
    print(func_str)
"""

'\n# --- Generate the 8 Functions ---\n\nfunctions = {}\nfor i in range(3):\n    dataset_file = f"data/problem_{i}.npz"\n    print(f"Processing dataset {i} from {dataset_file}...")\n    expr = run_symbolic_regression(dataset_file, population_size=100, generations=500, max_depth=5)\n    # You might want to post-process the expression for aesthetics.\n    func_def = f"def f{i+1}(x: np.ndarray) -> np.ndarray:\n"\n    func_def += f"    return {expr}\n"\n    functions[f\'f{i+1}\'] = func_def\n\n# Print the functions\nprint("\nGenerated Functions:")\nfor key, func_str in functions.items():\n    print(func_str)\n'

In [113]:
# --- Generate Function f1 ---
dataset_file = "data/problem_1.npz"
print(f"Processing dataset 1 from {dataset_file}...")
expr = run_symbolic_regression(dataset_file, population_size=100, generations=500, max_depth=5)
def f1(x: np.ndarray) -> np.ndarray:
    return {expr}
print("\nGenerated function f1:")
print("def f1(x: np.ndarray) -> np.ndarray:")
print(f"    return {expr}")

Processing dataset 1 from data/problem_1.npz...
Generation 0: best fitness = 0.0025488196979635298
Early stopping at generation 2: best fitness = 0.0
Best expression found:
sin(x[0])

Generated function f1:
def f1(x: np.ndarray) -> np.ndarray:
    return sin(x[0])


In [114]:
# --- Generate Function f2 ---
dataset_file = "data/problem_2.npz"
print(f"Processing dataset 2 from {dataset_file}...")
expr = run_symbolic_regression(dataset_file, population_size=100, generations=500, max_depth=5)
def f2(x: np.ndarray) -> np.ndarray:
    return {expr}
print("\nGenerated function f2:")
print("def f2(x: np.ndarray) -> np.ndarray:")
print(f"    return {expr}")

Processing dataset 2 from data/problem_2.npz...
Generation 0: best fitness = 29616967656310.176
Generation 100: best fitness = 28730857173309.824
Generation 200: best fitness = 26664622647569.062
Generation 300: best fitness = 19156001503386.55
Generation 400: best fitness = 19156001503386.55
Generation 499: best fitness = 19156001503386.55
Best expression found:
(x[0] / ((((x[1] - x[1]) + (2.37 * x[2])) / sin(tan(x[2]))) / ((sin(x[0]) + x[1]) * tan((-2.81 / -2.81)))))

Generated function f2:
def f2(x: np.ndarray) -> np.ndarray:
    return (x[0] / ((((x[1] - x[1]) + (2.37 * x[2])) / sin(tan(x[2]))) / ((sin(x[0]) + x[1]) * tan((-2.81 / -2.81)))))


In [115]:
# --- Generate Function f3 ---
dataset_file = "data/problem_3.npz"
print(f"Processing dataset 3 from {dataset_file}...")
expr = run_symbolic_regression(dataset_file, population_size=100, generations=500, max_depth=5)
def f3(x: np.ndarray) -> np.ndarray:
    return {expr}
print("Generated function f3:")
print("def f3(x: np.ndarray) -> np.ndarray:")
print(f"    return {expr}")

Processing dataset 3 from data/problem_3.npz...
Generation 0: best fitness = 2892.8269606996864
Generation 100: best fitness = 1267.9394707177937
Generation 200: best fitness = 1267.9394707177937
Generation 300: best fitness = 1267.9394707177937
Generation 400: best fitness = 1209.8309913478174
Generation 499: best fitness = 1209.8309913478174
Best expression found:
(exp((0.36 - x[1])) + sin(0.36))
Generated function f3:
def f3(x: np.ndarray) -> np.ndarray:
    return (exp((0.36 - x[1])) + sin(0.36))


In [ ]:
# --- Generate Function f4 ---
dataset_file = "data/problem_4.npz"
print(f"Processing dataset 4 from {dataset_file}...")
expr = run_symbolic_regression(dataset_file, population_size=100, generations=500, max_depth=5)
def f4(x: np.ndarray) -> np.ndarray:
    return {expr}
print("Generated function f4:")
print("def f4(x: np.ndarray) -> np.ndarray:")
print(f"    return {expr}")

In [ ]:
# --- Generate Function f5 ---
dataset_file = "data/problem_5.npz"
print(f"Processing dataset 5 from {dataset_file}...")
expr = run_symbolic_regression(dataset_file, population_size=100, generations=500, max_depth=5)
def f5(x: np.ndarray) -> np.ndarray:
    return {expr}
print("Generated function f5:")
print("def f5(x: np.ndarray) -> np.ndarray:")
print(f"    return {expr}")

In [ ]:
# --- Generate Function f6 ---
dataset_file = "data/problem_6.npz"
print(f"Processing dataset 6 from {dataset_file}...")
expr = run_symbolic_regression(dataset_file, population_size=100, generations=500, max_depth=5)
def f6(x: np.ndarray) -> np.ndarray:
    return {expr}
print("Generated function f6:")
print("def f6(x: np.ndarray) -> np.ndarray:")
print(f"    return {expr}")

In [ ]:
# --- Generate Function f7 ---
dataset_file = "data/problem_7.npz"
print(f"Processing dataset 7 from {dataset_file}...")
expr = run_symbolic_regression(dataset_file, population_size=100, generations=500, max_depth=5)
def f7(x: np.ndarray) -> np.ndarray:
    return {expr}
print("Generated function f7:")
print("def f7(x: np.ndarray) -> np.ndarray:")
print(f"    return {expr}")

In [ ]:
# --- Generate Function f8 ---
dataset_file = "data/problem_8.npz"
print(f"Processing dataset 8 from {dataset_file}...")
expr = run_symbolic_regression(dataset_file, population_size=100, generations=500, max_depth=5)
def f8(x: np.ndarray) -> np.ndarray:
    return {expr}
print("Generated function f8:")
print("def f8(x: np.ndarray) -> np.ndarray:")
print(f"    return {expr}")